# Visualize Change Masks

Bird's-eye view of frame-to-frame change masks across sequences.
For each sequence, displays a strip with raw grayscale frames on top
and corresponding binary change masks below.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from mtb_change_detection.change_detector import compute_change_mask
from mtb_change_detection.data import (
    get_sorted_frames,
    is_wf_sequence,
    list_sequences,
    load_label_boxes,
)
from mtb_change_detection.detector import load_inference_results

## Configuration

In [ ]:
SPLIT = "val"  # "train" or "val"
PIXEL_THRESHOLD = 10  # from params.yaml — try 19 for paper default
MAX_SEQUENCES = 20  # how many sequences to display
CATEGORIES = ["wildfire", "fp"]  # filter by category, or None for all
# Set to a list of names to pick specific sequences
SEQUENCE_NAMES = None  # e.g. ["seq_1", "seq_2"]

DATA_DIR = Path("../data/01_raw/datasets") / SPLIT
INFER_DIR = Path("../data/02_intermediate") / SPLIT

## Select sequences

In [ ]:
sequences = list_sequences(DATA_DIR)

if CATEGORIES:
    sequences = [s for s in sequences if s.parent.name in CATEGORIES]
if SEQUENCE_NAMES:
    sequences = [s for s in sequences if s.name in SEQUENCE_NAMES]

sequences = sequences[:MAX_SEQUENCES]

print(f"Selected {len(sequences)} sequences:")
for seq in sequences:
    label = "wildfire" if is_wf_sequence(seq) else "fp"
    n_frames = len(get_sorted_frames(seq))
    print(f"  {seq.parent.name}/{seq.name}  ({label}, {n_frames} frames)")

## Visualization

In [ ]:
def overlay_change_mask(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Return an RGB image with changed pixels highlighted in red."""
    rgb = np.stack([gray, gray, gray], axis=-1)
    rgb[mask, 0] = 255  # red channel
    rgb[mask, 1] = 0
    rgb[mask, 2] = 0
    return rgb


def draw_boxes(ax, detections, img_h, img_w, color, linewidth=2):
    """Draw normalized center-based bboxes on a matplotlib axis."""
    for det in detections:
        x = (det.cx - det.w / 2) * img_w
        y = (det.cy - det.h / 2) * img_h
        w = det.w * img_w
        h = det.h * img_h
        rect = Rectangle(
            (x, y),
            w,
            h,
            linewidth=linewidth,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)


def plot_sequence_strip(
    sequence_dir: Path,
    pixel_threshold: int,
    infer_dir: Path,
) -> None:
    """Display a 2-row strip with bboxes: GT (green), preds (purple)."""
    frame_paths = get_sorted_frames(sequence_dir)
    if len(frame_paths) < 2:
        print(f"Skipping {sequence_dir.name}: < 2 frames")
        return

    n = len(frame_paths)
    label = "wildfire" if is_wf_sequence(sequence_dir) else "fp"

    # Print path above the figure so it stays visible on interaction
    seq_path = sequence_dir.resolve()
    print(
        f"\n{'=' * 80}\n"
        f"{seq_path}  [{label}]  (threshold={pixel_threshold})\n"
        f"{'=' * 80}"
    )

    # Load YOLO predictions keyed by frame_id
    infer_path = infer_dir / f"{sequence_dir.name}.json"
    preds_by_frame = {}
    if infer_path.is_file():
        for fr in load_inference_results(infer_path):
            preds_by_frame[fr.frame_id] = fr.detections

    # Load all frames (color for display, grayscale for diffing)
    colors = []
    grays = []
    for p in frame_paths:
        bgr = cv2.imread(str(p))
        if bgr is None:
            print(f"  Warning: could not read {p.name}")
            return
        colors.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        grays.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY))

    # Compute change masks for consecutive pairs
    masks = [None]
    for i in range(1, n):
        if grays[i - 1].shape != grays[i].shape:
            masks.append(np.zeros_like(grays[i], dtype=bool))
        else:
            masks.append(compute_change_mask(grays[i - 1], grays[i], pixel_threshold))

    # Plot
    thumb_w = 7
    fig, axes = plt.subplots(
        2,
        n,
        figsize=(thumb_w * n, thumb_w * 2),
        gridspec_kw={"hspace": 0.12, "wspace": 0.05},
    )
    if n == 1:
        axes = axes.reshape(2, 1)

    img_h, img_w = grays[0].shape

    for i in range(n):
        frame_id = frame_paths[i].stem

        # Top row: color frame + GT (green) + predictions (purple)
        axes[0, i].imshow(colors[i])

        label_path = sequence_dir / "labels" / f"{frame_id}.txt"
        gt_boxes, _ = load_label_boxes(label_path)
        draw_boxes(axes[0, i], gt_boxes, img_h, img_w, "lime")

        pred_boxes = preds_by_frame.get(frame_id, [])
        draw_boxes(axes[0, i], pred_boxes, img_h, img_w, "mediumorchid")

        ts = frame_id.split("T")[-1].replace("-", ":")
        axes[0, i].set_title(ts, fontsize=11)
        axes[0, i].axis("off")

        # Bottom row: change overlay (no boxes)
        if masks[i] is None:
            axes[1, i].imshow(grays[i], cmap="gray", vmin=0, vmax=255)
            axes[1, i].set_title("(no prev)", fontsize=11, color="gray")
        else:
            axes[1, i].imshow(overlay_change_mask(grays[i], masks[i]))
            changed_pct = 100.0 * masks[i].sum() / masks[i].size
            axes[1, i].set_title(f"{changed_pct:.1f}%", fontsize=11)
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Frame", fontsize=13)
    axes[1, 0].set_ylabel("Change", fontsize=13)

    plt.show()

In [ ]:
for seq_dir in sequences:
    plot_sequence_strip(seq_dir, PIXEL_THRESHOLD, INFER_DIR)